## Dataset

This exercise uses the UCI Student Performance dataset. Download `student-mat.csv` from the official UCI source and place it at `data/student-mat.csv` before running the notebook.

Source: https://archive.ics.uci.edu/dataset/320/student+performance


# Machine Learning Regression Pipeline — Student Performance

## Objective

Build a simple regression pipeline from scratch and learn the complete flow:

**Dataset → Inspection → Feature/Target Selection → EDA → Train/Test Split → Linear Regression → Prediction → Evaluation → Feature Comparison**

This experiment deliberately uses only numerical features (`G1` and `G2`) so that we can focus on the machine-learning pipeline before introducing categorical preprocessing.


## 1. Import Libraries

We use:

- **Pandas** for loading and inspecting tabular data.
- **NumPy** for numerical operations.
- **Matplotlib** for simple visual exploration.
- **Scikit-learn** for splitting the data, training Linear Regression, and evaluating predictions.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("data/student-mat.csv", sep=";")


## 2. Initial Dataset Inspection

These are our standard first checks when no special EDA requirements are given:

- `head()` — see sample rows.
- `shape` — understand dataset size.
- `info()` — inspect columns, data types, and missing values.
- `describe()` — inspect numerical distributions.
- `isnull().sum()` — explicitly check missing values.

We use the results to decide what additional EDA is actually necessary.


In [ ]:
df.head()
print("Shape:", df.shape)
df.info()
print("\nDescriptive statistics:")
display(df.describe())
print("\nMissing values:")
print(df.isnull().sum())


## 3. Define the Prediction Target

The target is `G3`, the student's final grade.

For this first experiment we intentionally choose only two numerical features:

- `G1` — first-period grade
- `G2` — second-period grade

This lets us compare:

1. `G1` only
2. `G2` only
3. `G1 + G2`

before introducing more features or categorical preprocessing.


In [ ]:
target = "G3"

X = df[["G1", "G2"]]
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures:", X.columns.tolist())
print("Target:", target)


## 4. Target Exploration

Before training, inspect the target itself.

Here we look at:

- numerical summary
- frequency of each final grade

This helps us understand the range and distribution of what the model is trying to predict.


In [ ]:
print(y.describe())
print("\nFinal grade counts:")
print(y.value_counts().sort_index())


## 5. Simple EDA — Feature vs Target

Because our experiment is specifically about predicting `G3` from `G1` and `G2`, the most useful first visual check is whether these features have a visible relationship with the target.

We do not need to create every possible plot. We choose plots that help answer the modeling question.


In [ ]:
plt.scatter(df["G1"], df["G3"])
plt.xlabel("G1")
plt.ylabel("G3")
plt.title("G1 vs G3")
plt.show()


In [ ]:
plt.scatter(df["G2"], df["G3"])
plt.xlabel("G2")
plt.ylabel("G3")
plt.title("G2 vs G3")
plt.show()


## 6. Train/Test Split

We keep part of the data unseen during training.

- **Training set** — used by the model to learn its parameters.
- **Test set** — used after training to estimate performance on unseen data.

`random_state=42` makes the split reproducible.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


## 7. Linear Regression

Linear Regression learns a relationship between the input features and the numerical target.

For two features:

ŷ = β₀ + β₁G1 + β₂G2

where:

- `ŷ` is the predicted final grade.
- `β₀` is the intercept.
- `β₁` and `β₂` are learned coefficients.

The model learns these parameters from the training data.


In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)


## 8. Make Predictions

After training, the model can predict `G3` for the test data.

The test data was not used to learn the coefficients, so it gives us an initial check of generalization.


In [ ]:
y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    "Actual_G3": y_test.values,
    "Predicted_G3": y_pred
})

comparison.head(10)


## 9. Model Evaluation

We use several metrics because each tells us something different.

- **Mean Absolute Error (MAE)** — average absolute prediction error.
- **Mean Squared Error (MSE)** — average squared prediction error.
- **Root Mean Squared Error (RMSE)** — square root of MSE, expressed in the target's units.
- **R-squared (R²)** — measures the proportion of target variance explained by the model relative to a mean-prediction baseline.

There is no universal cutoff that makes an R² value "good" or "bad". We interpret metrics in the context of the problem and by comparing appropriate models.


In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("R-squared (R²):", r2)


## 10. Understand the Learned Equation

The coefficients show how the fitted Linear Regression equation uses the features.

We can also manually substitute feature values into the learned equation to understand what `predict()` is doing internally.


In [ ]:
print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)

prediction = (
    model.intercept_
    + model.coef_[0] * 12
    + model.coef_[1] * 14
)

print("Manual prediction for G1=12, G2=14:", prediction)


## 11. Experiment — G1 vs G2 vs G1 + G2

Now we test a simple model-selection question:

**Does using both intermediate grades give a better prediction than using either one alone?**

We train the same Linear Regression algorithm three times and compare the metrics.

The important lesson is that we are not deciding based on a memorized rule. We are creating comparable experiments and using evidence from the results.


In [ ]:
def train_and_evaluate(features):
    X = df[features]
    y = df["G3"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    return model, mae, mse, rmse, r2


In [ ]:
model_g1, mae_g1, mse_g1, rmse_g1, r2_g1 = train_and_evaluate(["G1"])

print("G1 only")
print("Coefficient:", model_g1.coef_)
print("Intercept:", model_g1.intercept_)
print("Mean Absolute Error (MAE):", mae_g1)
print("Root Mean Squared Error (RMSE):", rmse_g1)
print("R-squared (R²):", r2_g1)


In [ ]:
model_g2, mae_g2, mse_g2, rmse_g2, r2_g2 = train_and_evaluate(["G2"])

print("G2 only")
print("Coefficient:", model_g2.coef_)
print("Intercept:", model_g2.intercept_)
print("Mean Absolute Error (MAE):", mae_g2)
print("Root Mean Squared Error (RMSE):", rmse_g2)
print("R-squared (R²):", r2_g2)


In [ ]:
model_both, mae_both, mse_both, rmse_both, r2_both = train_and_evaluate(["G1", "G2"])

print("G1 + G2")
print("Coefficients:", model_both.coef_)
print("Intercept:", model_both.intercept_)
print("Mean Absolute Error (MAE):", mae_both)
print("Root Mean Squared Error (RMSE):", rmse_both)
print("R-squared (R²):", r2_both)


## 12. Compare the Experiments

Putting the results together makes the model comparison explicit.

For error metrics:

- lower **Mean Absolute Error (MAE)** is better
- lower **Root Mean Squared Error (RMSE)** is better

For R-squared:

- higher **R²** is better, all else being equal

But the final model choice should also consider the problem, data, complexity, interpretability, and validation results.


In [ ]:
results = pd.DataFrame({
    "Features": ["G1", "G2", "G1 + G2"],
    "Mean Absolute Error (MAE)": [mae_g1, mae_g2, mae_both],
    "Root Mean Squared Error (RMSE)": [rmse_g1, rmse_g2, rmse_both],
    "R-squared (R²)": [r2_g1, r2_g2, r2_both]
})

results


## 13. What This Experiment Taught

This first experiment established the basic supervised-learning regression pipeline:

**Load → Inspect → Select features/target → Explore → Split → Train → Predict → Evaluate → Compare**

Key lessons:

1. Start with a simple baseline.
2. Keep the test data separate from training.
3. Choose EDA based on the question and data, not by blindly running every possible plot.
4. Use multiple evaluation metrics.
5. Compare models/features using evidence rather than arbitrary universal cutoffs.
6. A model with poor absolute performance is not automatically proven to be underfitting; compare it with appropriate alternatives.
7. Feature selection itself can be treated as an experiment.

The next experiment will build on this pipeline rather than starting from zero.
